In [24]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt
import os
import urllib.request
import zipfile
import numpy as np


print("TensorFlow versione:", tf.__version__)

TensorFlow versione: 2.20.0


In [25]:
# Scaricare i dataset
print("Scaricando training set...")
urllib.request.urlretrieve(
    "https://storage.googleapis.com/download.tensorflow.org/data/rps.zip",
    "rps.zip"
)

print("Scaricando test set...")
urllib.request.urlretrieve(
    "https://storage.googleapis.com/download.tensorflow.org/data/rps-test-set.zip",
    "rps-test-set.zip"
)

print("Estraendo...")
with zipfile.ZipFile("rps.zip", "r") as z:
    z.extractall(".")
with zipfile.ZipFile("rps-test-set.zip", "r") as z:
    z.extractall(".")

print("Dataset pronti!")

Scaricando training set...
Scaricando test set...
Estraendo...
Dataset pronti!


In [26]:
# Percorsi delle cartelle
TRAIN_DIR = "./rps"
TEST_DIR  = "./rps-test-set"

# Generatore TRAINING con augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,          # Normalizzazione
    rotation_range=40,       # Ruota fino a 40 gradi
    width_shift_range=0.2,   # Sposta orizzontalmente
    shear_range=0.2,         # Deforma
    horizontal_flip=True,    # Specchia
    fill_mode='nearest'      # Riempie i pixel vuoti
)

# Generatore TEST (solo normalizzazione, niente augmentation)
test_datagen = ImageDataGenerator(rescale=1./255)

# Collega i generatori alle cartelle
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(150, 150),  # Ridimensiona ogni immagine a 150x150
    batch_size=32,
    class_mode='categorical' # 3 classi: rock, paper, scissors
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(150, 150),
    batch_size=32,
    class_mode='categorical'
)

print("Classi trovate:", train_generator.class_indices)

Found 2520 images belonging to 3 classes.
Found 372 images belonging to 3 classes.
Classi trovate: {'paper': 0, 'rock': 1, 'scissors': 2}


In [ ]:
# Alcune immagini per visualizzare il processo
sample_images, sample_labels = next(train_generator)

plt.figure(figsize=(10, 5))
for i in range(6):
    plt.subplot(2, 3, i+1)
    plt.imshow(sample_images[i])
    plt.axis('off')
plt.suptitle("Esempi di immagini con Augmentation")
plt.show()

In [28]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout

model = Sequential([

    # --- BLOCCO 1 ---
    # 32 filtri cercano pattern semplici (bordi, linee)
    Conv2D(32, (3,3), activation='relu', input_shape=(150, 150, 3)),
    MaxPooling2D(2, 2),

    # --- BLOCCO 2 ---
    # 64 filtri cercano pattern più complessi (curve, angoli)
    Conv2D(64, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # --- BLOCCO 3 ---
    # 128 filtri cercano pattern ancora più complessi (forme di dita)
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # --- BLOCCO 4 ---
    # 128 filtri per affinare ulteriormente
    Conv2D(128, (3,3), activation='relu'),
    MaxPooling2D(2, 2),

    # --- PARTE FINALE ---
    Flatten(),           # Srotola tutto in una lista
    Dense(512, activation='relu'),  # 512 neuroni per "ragionare"
    Dense(3, activation='softmax')  # 3 neuroni = 3 classi finali
])

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d_12 (Conv2D)              │ (None, 148, 148, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_12 (MaxPooling2D) │ (None, 74, 74, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_13 (Conv2D)              │ (None, 72, 72, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_13 (MaxPooling2D) │ (None, 36, 36, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_14 (Conv2D)              │ (None, 34, 34, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_14 (MaxPooling2D) │ (None, 17, 17, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 15, 15, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_15 (MaxPooling2D) │ (None, 7, 7, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_3 (Flatten)             │ (None, 6272)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 512)            │     3,211,776 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 3)              │         1,539 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,454,147 (13.18 MB)

 Trainable params: 3,454,147 (13.18 MB)

 Non-trainable params: 0 (0.00 B)

In [29]:
# Strategia di apprendimento
model.compile(
    optimizer='adam',                    # Algoritmo di ottimizzazione
    loss='categorical_crossentropy',     # Funzione di errore per 3+ classi
    metrics=['accuracy']                 # Monitora l'accuratezza durante il training
)

print("Modello compilato, pronto per il training!")

Modello compilato, pronto per il training!


In [30]:
from tensorflow.keras.callbacks import EarlyStopping

# EarlyStopping: interrompe il training se accuracy supera 98%
early_stop = EarlyStopping(
    monitor='accuracy',      # Controlla l'accuratezza sul training set
    patience=3,              # Aspetta 3 epoche prima di fermarsi
    verbose=1                # Stampa un messaggio quando si ferma
)

# Avvia il training!
history = model.fit(
    train_generator,         # Dati di training con augmentation
    epochs=20,               # Massimo 20 epoche
    validation_data=test_generator,   # Valuta su test set ad ogni epoca
    callbacks=[early_stop]   # Usa l'EarlyStopping
)

Epoch 1/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 31s 342ms/step - accuracy: 0.5409 - loss: 0.9185 - val_accuracy: 0.8118 - val_loss: 0.4884
Epoch 2/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 24s 302ms/step - accuracy: 0.9063 - loss: 0.2590 - val_accuracy: 0.9570 - val_loss: 0.1324
Epoch 3/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 24s 301ms/step - accuracy: 0.9635 - loss: 0.1155 - val_accuracy: 0.9489 - val_loss: 0.1547
Epoch 4/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 23s 292ms/step - accuracy: 0.9544 - loss: 0.1457 - val_accuracy: 0.9704 - val_loss: 0.0946
Epoch 5/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 23s 291ms/step - accuracy: 0.9706 - loss: 0.0846 - val_accuracy: 0.9220 - val_loss: 0.1512
Epoch 6/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 24s 300ms/step - accuracy: 0.9841 - loss: 0.0564 - val_accuracy: 0.9704 - val_loss: 0.0776
Epoch 7/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 24s 298ms/step - accuracy: 0.9865 - loss: 0.0388 - val_accuracy: 0.8656 - val_loss: 0.2827
Epoch 8/20
79/79 ━━━━━━━━━━━━━━━━━━━━ 24s 299ms/step - accuracy: 0.9901 - loss: 0.0316 - val_accu

In [ ]:
# Visualizza l'andamento del training
acc = history.history['accuracy']
val_acc = history.history['val_accuracy']
loss = history.history['loss']
val_loss = history.history['val_loss']

epochs_range = range(len(acc))

plt.figure(figsize=(12, 4))

# Grafico accuratezza
plt.subplot(1, 2, 1)
plt.plot(epochs_range, acc, label='Training Accuracy')
plt.plot(epochs_range, val_acc, label='Validation Accuracy')
plt.legend()
plt.title('Accuratezza nel tempo')

# Grafico loss
plt.subplot(1, 2, 2)
plt.plot(epochs_range, loss, label='Training Loss')
plt.plot(epochs_range, val_loss, label='Validation Loss')
plt.legend()
plt.title('Errore nel tempo')

plt.show()